# 2.1 Simframe

[Simframe](https://simframe.readthedocs.io/en/latest/) is a Python framework for scientific simulations. Since DustPy is based on Simframe this notebook will be a short and concise introduction on the usage and philosophy of Simframe.

Simframe allows you to easily integrate differential equations by simply providing a function returning the derivative of a quantity. The advantage is that you do not have to care about peripherals such as organizing the structure of your project or writing and reading data files. These things will be dealt with by Simframe under the hood.

Consider the following simple problem that we want to solve. Given the differential equation

$\Large \frac{\mathrm{d}}{\mathrm{d}t} Y = -bY$

with the initial condition 

$\Large Y\left( t=0 \right) = A$.

The solution of this problem is known

$\Large Y\left( t \right) = A  e^{-bt}$.

Additionally, we do not only want to integrate the quantity $Y$, but we want to also compute the quantity $T$, which is derived from $Y$ with

$\Large T = - \frac{1}{b} \log \frac{Y}{A}$,

This has the simply solution $T=t$ which we can compare later.

## Setting up the model

In [2]:
from simframe import Frame

In [3]:
sim = Frame(description="Simple example")

In the first step we create a Simframe object, which is an empty frame which needs to be filled by us.

In [4]:
sim

Frame (Simple example)
----------------------
    Integrator   : not specified
    Writer       : not specified

### Structuring the simulation frame

In this step we create a group to store the simulation parameters $A$, $b$, as well as or desired time step of the integration $\Delta t$.

In [8]:
sim.addgroup("pars", description="Simulation parameters");

In [9]:
sim

Frame (Simple example)
----------------------
    pars         : Group (Simulation parameters)
  -----
    Integrator   : not specified
    Writer       : not specified

Then we add the parameters to the group.

In [10]:
sim.pars.addfield("A", 10., description="Initial value of Y", constant=True);
sim.pars.addfield("b", 1., description="Decay rate", constant=True);
sim.pars.addfield("dt", 0.5, description="Time step");

In [11]:
sim.pars

Group (Simulation parameters)
-----------------------------
    A            : Field (Initial value of Y), constant
    b            : Field (Decay rate), constant
    dt           : Field (Time step)
  -----

These parameters can be easily addressed just like NumPy arrays.

In [12]:
sim.pars.A

10.0

In fact, they are NumPy arrays.

In [16]:
import numpy as np

In [17]:
isinstance(sim.pars.A, np.ndarray)

True

### Adding the integration variable

In the next step we add the integration variable $t$.

In [18]:
sim.addintegrationvariable("t", 0., description="Time");

In [19]:
sim

Frame (Simple example)
----------------------
    pars         : Group (Simulation parameters)
  -----
    t            : IntVar (Time), Integration variable
  -----
    Integrator   : not specified
    Writer       : not specified

The integration variable needs two additional pieces of information:  
It needs a prescription to calculate the time step it should take from the current state of the simulation frame. And it needs to know when to write output files.

To compute the time step we have to provide a function that takes the simulation frame as input and returns the timestep. In this example we simply want to return the time step stored in the parameters group.

In [20]:
def dt(sim):
    return sim.pars.dt

This function it then assigned to the so-called "updater" of the integration variable, because it tells the variable how to advance itself during an integration step. 

In [21]:
sim.t.updater = dt

Now we tell the integration variable at which values it should write output files. The final value is also the final time of the simulation.

In [22]:
sim.t.snapshots = np.arange(0., 11., 1.)

### Adding fields to the simulation frame

Next we are going to add fields for the variables $Y$ and $T$ and initialize them with their respective values.

In [24]:
sim.addfield("Y", sim.pars.A, description="Integrated variable");
sim.addfield("T", 0., description="Updated variable");

Both variables are conceptually different: the first one is being integrated, while the second one is simply updated from the current state of the simulation frame.  
In the latter case we have to provide a function that takes the simulation frame as input and returns the new values of the variable.

In [34]:
def T(sim):
    """
    Function to compute `T` from `Y` and `b`.
    """
    return -np.log(sim.Y/sim.pars.A) / sim.pars.b

As previously, this function needs to be added to the updater of the field.

In [35]:
sim.T.updater = T

Additionally, we need to tell Simframe that it has to update this field once after every integration step. This can be done by providing the updater of the simulation frame with a list of the field names to be updated in the correct order. In this case we only have one field $T$ to be updated.

In [36]:
sim.updater = ["T"]

In [37]:
sim.T.updater

Heartbeat
---------

Systole:  None
Updater:  <function T at 0x7fd095552f00>
Diastole: None

Docstrings
----------

Systole:
The type of the None singleton.

Updater:

Function to compute `T` from `Y` and `b`.


Diastole:
The type of the None singleton.

### Adding differential equations

The field $Y$ is not updated, but integrated. We therefore have to provide a function that returns the derivative of the field. This function needs in addition to the simulation frame also the integration variable and the field itself as input parameters.

In [19]:
def dYdt(sim, t, Y):
    return -sim.pars.b*Y

Instead of adding this function the updater, we have to add it to the "differentiator" of the field.

In [20]:
sim.Y.differentiator = dYdt

### Setting up the integrator

In the next step we need to add an integrator to the simulation, which tells Simframe what needs to be integrated and how to do it. The integrator needs to be initialized with the integration variable that is relevant for this problem.

In [21]:
from simframe import Integrator

In [22]:
sim.integrator = Integrator(sim.t)

And we need to give the integrator a list of integration instructions that tell it, what variable needs to be integrated with which integration scheme. In this example we are using a simple explicit Euler first order scheme.

In [23]:
from simframe import Instruction
from simframe import schemes

In [24]:
sim.integrator.instructions = [
    Instruction(schemes.expl_1_euler, sim.Y)
]

### Configuring the writer

In the last step we can tell Simframe how to writer output files. This is not strictly necessary. We could also run the simulation without writing any files. In this example we want to writer output files in the .hdf5 file format.

In [25]:
from simframe import writers

In [26]:
sim.writer = writers.hdf5writer()

We change the name of the datadirectory and allow Simframe to overwrite existing files.

In [27]:
sim.writer.datadir = "simframe"
sim.writer.overwrite = True

## Running the simulation

The simulation frame is now fully configured.

In [28]:
sim

Frame (Simple example)
----------------------
    pars         : Group (Simulation parameters)
  -----
    t            : IntVar (Time), Integration variable
    T            : Field (Updated variable)
    Y            : Field (Integrated variable)
  -----
    Integrator   : Integrator
    Writer       : Writer (HDF5 file format using h5py)

We can start the simulation.